# Setup

In [5]:
import os, sys

SUMO_HOME  = r'C:\Program Files (x86)\Eclipse\Sumo'
SUMO_HOME  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/sumo'
# PROJ_PATH  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/framework/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/proj'


os.environ['SUMO_HOME']  = SUMO_HOME
# os.environ['PROJ_LIB']   = PROJ_PATH
# os.environ['PROJ_DATA']  = PROJ_PATH

sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# Verify all three
checks = {
    'SUMO_HOME':  os.path.exists(SUMO_HOME),
#     'proj.db':    os.path.exists(os.path.join(PROJ_PATH, 'proj.db')),
    'gtfs2pt.py': os.path.exists(f'{SUMO_HOME}/tools/import/gtfs/gtfs2pt.py'),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")

SUMO_HOME      : ✅
gtfs2pt.py     : ✅


In [11]:
# cd /Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2
# !cd "C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS"
# !cd


os.getcwd()

'/Users/soheil/codes/Simulation/DowntownSeattleSUMO/Simulation/GTFS'

# OSM Map Extraction

-extraction from Geofrabil for Full Washington Link Network

In [24]:
import requests

# Download Washington state extract from Geofabrik
url = "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf"
out = "washington.osm.pbf"

print("Downloading Washington state from Geofabrik (~100MB)...")
with requests.get(url, stream=True) as r:
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(out, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"\r  {pct:.1f}% ({downloaded/1024/1024:.1f} MB)", end='')

print(f"\n✅ Done — saved to {out}")

  100.0% (339.4 MB)
✅ Done — saved to washington.osm.pbf


In [52]:
!osmium extract \
  -b -122.4500,47.2500,-122.1000,47.8500 \
  washington.osm.pbf \
  -o seattle_link.osm.pbf \
  --strategy complete_ways

No extract specified in config file or on the command line.


In [56]:
!osmium extract \
  -b -124.8,45.5,-116.9,49.1 \
  washington.osm.pbf \
  -o washington_full.osm.pbf \
  --strategy complete_ways

[======================================================================] 100% 


In [36]:
!osmium cat seattle_link.osm.pbf -o seattle_link.osm.xml

[======================================================================] 100% 


In [58]:
!osmium cat washington_full.osm.pbf -o seattle_link_full.osm.xml

[======================================================================] 100% 


# Network Preparation

In [ ]:
!netconvert \
  --osm-files seattle_link.osm.xml \
  -o seattle_lightrail_3.net.xml \
  --type-files $SUMO_HOME/data/typemap/osmNetconvert.typ.xml,$SUMO_HOME/data/typemap/osmNetconvertRailUsage.typ.xml \
  --keep-edges.by-type railway.light_rail,railway.subway \
  --proj.utm true \
  --geometry.remove \
  --junctions.join \
  --output.street-names \
  --ptstop-output seattle_rail_stops.add.xml \
  --ptline-output seattle_rail_ptlines.add.xml \
  --osm.stop-output.length 30

### Converging new Lightrail Segments

In [ ]:
!netconvert -s soheil_seattle.net.xml --remove-edges.by-vclass rail,rail_urban,rail_fast,subway,rail_electric -o soheil_seattle_veh.net.xml

In [ ]:
!netconvert -s seattle_lightrail_2.net.xml,soheil_seattle_veh.net.xml -o merged.net.xml 

#### Adding Full Lightrail Network

In [5]:
# !netconvert -s "Zili\rail\toy network\seattle_lightrail_DT_excluded.net.xml",soheil_seattle_correct_premissions.net.xml  -o soheil_seattle_merged_v2.net.xml 

Success.


In [5]:
!netconvert -s soheil_seattle_correct_premissions.net.xml,"Zili\rail\toy network\seattle_lightrail_DT_excluded.net.xml" -o soheil_seattle_merged.net.xml 

Success.


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a tram.add.xml,gtfs_pt_stops_fixed.add.xml,gtfs_pt_vehicles.add.xml


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a bus.add.xml,gtfs_pt_stops.add.xml,gtfs_pt_vehicles.add.xml

# fix speeds of the network. 
used the code from run_simulation.py

In [2]:
import os, sys, subprocess
sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))
import sumolib
from sumolib.net.lane import SUMO_VEHICLE_CLASSES

NET_IN  = "soheil_seattle_merged.net.xml"
NET_OUT = "soheil_seattle_merged_pedspeed.net.xml"
PATCH   = "ped_speed_patch.edg.xml"

SMALL_SPEED_LIMIT = 8.94   # m/s (20 mph) -- used when the edge has no "_1" lane
REF_LANE_INDEX    = 1
ALL_VCLASSES      = len(SUMO_VEHICLE_CLASSES)

net = sumolib.net.readNet(NET_IN)

patch = {}
for edge in net.getEdges():
    lanes = edge.getLanes()
    ref = lanes[REF_LANE_INDEX] if len(lanes) > REF_LANE_INDEX else None
    for lane in lanes:
        # Same rule as traci: getAllowed() returns [] for a lane with no allow/disallow, so only
        # lanes with an EXPLICIT permission list count. Edge gneE144 shows all three cases:
        #   gneE144_0  allow="pedestrian"        -> patched, speed copied from gneE144_1
        #   gneE144_1  disallow="pedestrian ..." -> the reference lane
        #   gneE144_2  (no allow/disallow)       -> skipped, stays 13.89 m/s
        # Skipping _2 is deliberate: the 44 unrestricted lanes in this net are hand-drawn gneE*
        # edges from the rail merge (plus 155091955_0), i.e. 50 km/h travel lanes, not sidewalks.
        if not lane.allows("pedestrian") or len(lane.getPermissions()) == ALL_VCLASSES:
            continue
        speed = ref.getSpeed() if ref is not None else SMALL_SPEED_LIMIT
        if abs(speed - lane.getSpeed()) > 1e-6:      # ref may be this lane -> no-op, as in traci
            patch.setdefault(edge.getID(), []).append((lane.getIndex(), speed))

with open(PATCH, "w", encoding="utf-8") as f:
    f.write("<edges>\n")
    for eid in sorted(patch):
        f.write('    <edge id="%s">\n' % eid)
        for idx, spd in sorted(patch[eid]):
            f.write('        <lane index="%d" speed="%.2f"/>\n' % (idx, spd))
        f.write("    </edge>\n")
    f.write("</edges>\n")

# netconvert applies the patch and regenerates internal lanes/walkingareas consistently
subprocess.run([sumolib.checkBinary("netconvert"),
                "-s", NET_IN, "-e", PATCH, "-o", NET_OUT], check=True)

print("patched %d lanes on %d edges -> %s" %
      (sum(len(v) for v in patch.values()), len(patch), NET_OUT))

pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db


Success.
patched 2883 lanes on 2452 edges -> soheil_seattle_merged_pedspeed.net.xml
